In [ ]:
import os
import sys
sys.path.append("/workspaces/dev/app")
os.chdir("/workspaces/dev")

In [ ]:
import msgpack
import numpy as np
import base64
import requests
import librosa
from IPython.display import Audio
import json

In [ ]:
from util.util import compress_to_opus, np_to_wav

In [ ]:
SAMPLE_RATE = 16000
BASE_URL = "https://api.sangjeong.com:8080"
# BASE_URL = "http://localhost:8000"

# 오디오 로드 및 세그멘팅

In [ ]:
original_audio, sr = librosa.load(".data/boda.wav", sr=SAMPLE_RATE)
# original_audio, sr = librosa.load(".data/news_with_english.wav", sr=SAMPLE_RATE)

In [ ]:
total_samples = len(original_audio)

segments = []
pos = 0
while pos < total_samples:
    rand_len = int(np.random.normal(loc=4000, scale=400))
    rand_len = np.clip(rand_len, 3500, 4500)
    end = min(pos + rand_len, total_samples)

    chunk = original_audio[pos:end]
    segments.append(chunk)
    pos = end

# 오디오 출력해보기

In [ ]:
idx = 0

In [ ]:
audio = segments[idx]
idx += 1
Audio(audio, rate=SAMPLE_RATE)

# 유틸

In [ ]:
def dump_using_json(data:dict):
    text = json.dumps(data).encode("utf-8")
    length = len(text).to_bytes(4, byteorder="big")
    return length + text

def dump_using_msgpack(data:dict):
    text = msgpack.packb(data)
    length = len(text).to_bytes(4, byteorder="big")
    return length + text

In [ ]:
def dump_audio(audio:np.ndarray):
    if audio is None:
        return b''
    # opus
    # audio = np_to_wav(audio, SAMPLE_RATE)
    # audio, _ = compress_to_opus(audio)
    return audio.tobytes()

# /diarization

In [ ]:
# 예시 음성 데이터의 일부
refer_dict = {
    "0": [(148061, 339589), (338760, 436145), (433746, 517335)],
    "1": [(1013274, 1068548), (1071733, 1209101), (1209858, 1344749)],
    "2": [(2563542, 2662059), (4373060, 4438192), (4570430, 4625790)],
    "3": [(880324, 938541), (7042377, 7096457), (7106339, 7156845)],
    "4": [(944284, 1009464), (1802918, 1921616), (1921626, 2053013)],
}

#### /diarization/embed_stream

In [ ]:
embedding_refer_dict = {"0": [], "1": [], "2": [], "3": [], "4": []}
for idx, ts in refer_dict.items():
    for s in ts:
        audio = original_audio[s[0] : s[1]]
        bt = dump_audio(audio)
        res = requests.post(
            BASE_URL + "/diarization/embed_stream",
            params={"sample_rate": SAMPLE_RATE},
            data=bt,
            headers={"Content-Type": "application/octet-stream"},
        )
        embedding_refer_dict[idx].append(res.json()["embedding"])

In [ ]:
# 임베딩 데이터 확인
embedding = embedding_refer_dict["0"][0]
print(embedding)
print(len(embedding))
bytes_ = base64.b64decode(embedding)
print(len(bytes_))
embedding = np.frombuffer(bytes_, dtype=np.float32)
print(embedding.shape)

#### /diarization/embed_file

In [ ]:
embedding_refer_dict = {"0": [], "1": [], "2": [], "3": [], "4": []}
for idx, ts in refer_dict.items():
    for s in ts:
        audio = original_audio[s[0] : s[1]]
        bt = dump_audio(audio)
        res = requests.post(
            BASE_URL + "/diarization/embed_file",
            params={"sample_rate": SAMPLE_RATE},
            files={"audio": ("audio.opus", bt, "application/octet-stream")},
        )
        embedding_refer_dict[idx].append(res.json()["embedding"])

In [ ]:
# 임베딩 데이터 확인
embedding = embedding_refer_dict["0"][0]
print(embedding)
print(len(embedding))
bytes_ = base64.b64decode(embedding)
print(len(bytes_))
embedding = np.frombuffer(bytes_, dtype=np.float32)
print(embedding.shape)

#### /diarization/refer_stream

In [ ]:
keys = list(embedding_refer_dict.keys())
bts = b''
for k in keys:
    for emb in embedding_refer_dict[k]:
        bts += base64.b64decode(emb)

res = requests.post(
    BASE_URL + "/diarization/refer_stream",
    params = {
        "group_id": "0",
        "user_id": "0", # 소켓에서는 필요 없다. http 통신에서는 어쩔 수 없이 넣어줘야 한다.
        "user_ids": keys,
        "counts": [len(embedding_refer_dict[k]) for k in keys],
    },
    data=bts,
)
result = res.json()
print(result)

#### /diarization/refer_file

In [ ]:
res = requests.post(
    BASE_URL + "/diarization/refer_file",
    params={
        "group_id": "0",
        "user_id": "0",
        "user_ids": keys,
        "counts": [len(embedding_refer_dict[k]) for k in keys]
    },
    files={"refers": ("audio.bin", bt, "application/octet-stream")},
)
result = res.json()
print(result)

#### /diarization/diarize_stream

In [ ]:
# 항상 diarize 요청전에, 해당 요청에 해당하는 사용자에 refer을 1개 이상 필수로 제공해야 함.
bt = dump_audio(segments[0])
res = requests.post(
    BASE_URL + "/diarization/diarize_stream",
    params={"group_id": "0", "user_id": "0", "sc_offset": 80000}, # sample rate offset을 넣으려면 sc_offset 파라미터 추가하면 된다.
    data=bt,
    headers={"Content-Type": "application/octet-stream"},
)
result = res.json()
print(result)

for segment in segments[1:100]:
    bt = dump_audio(segment)
    res = requests.post(
        BASE_URL + "/diarization/diarize_stream",
        params={"group_id": "0", "user_id": "0"}, # sample rate offset을 넣으려면 sc_offset 파라미터 추가하면 된다.
        data=bt,
        headers={"Content-Type": "application/octet-stream"},
    )
    result = res.json()
    print(result)

#### /diarization/diarize_file

In [ ]:
for segment in segments[100:200]:
    bt = dump_audio(segment)
    res = requests.post(
        BASE_URL + "/diarization/diarize_file",
        params={"group_id": "0", "user_id": "0"},
        files={"audio": ("audio.opus", bt, "application/octet-stream")},
    )
    result = res.json()
    print(result)

#### /llm/metadata

In [ ]:
res = requests.get(
    BASE_URL + "/llm/metadata",
    params={"group_id": "0"},
)

In [ ]:
res = requests.get(
    BASE_URL + "/llm/context",
    params={"group_id": "0"},
)
result = res.json()
print(result)

In [ ]:
res = requests.get(
    BASE_URL + "/llm/context_done",
    params={"group_id": "0"},
)
result = res.json()
print(result)